# 18b — Text-to-SQL (Gemini 2.5 Pro + AlloyDB)

**AWS-side equivalent:** Nova Pro + Aurora Serverless v2 (`notebooks/aws/18b-text-to-sql.ipynb`)
**GCP service:** Gemini 2.5 Pro structured output → AlloyDB via the AlloyDB Connector

Pattern (same shape as the AWS side):

```
Question → schema digest from information_schema → Gemini 2.5 Pro
         → JSON {sql, warnings} via response_schema
         → validator (SELECT/WITH only, allow-listed tables)
         → AlloyDB exec via read-only role
         → rows + grounded answer
```

_As of 2026-05, `google-genai` (`from google import genai`) is the supported Python client for Gemini structured output. Use `response_mime_type='application/json'` together with `response_schema` (Pydantic or dict). `gemini-2.5-pro` is required here — the reasoning headroom matters for joins and aggregations; `2.5-flash` underperforms on this prompt._

In [ ]:
# State plumbing — every notebook reads/writes config.json under the 'gcp' namespace.
import json
from pathlib import Path

config = json.loads(Path('config.json').read_text())
gcp = config.setdefault('gcp', {})
PROJECT_ID = gcp.get('project_id')
REGION = gcp.get('region', 'us-central1')
assert PROJECT_ID, 'Run 01-setup-credentials.ipynb first.'

TEXT2SQL_MODEL = gcp.get('text2sql_model', 'gemini-2.5-pro')
GROUNDING_MODEL = gcp.get('model_id', 'gemini-2.5-flash')

# AlloyDB connection variables — populated by 02-alloydb-graph.ipynb.
INSTANCE_URI = gcp.get('alloydb_instance_uri')
DB_NAME = gcp.get('alloydb_db', 'orgiq')
DB_USER = gcp.get('alloydb_user', 'postgres')
DB_PASSWORD_SECRET = gcp.get('alloydb_password_secret', 'orgiq-alloydb-password')
# Optional dedicated read-only role created by 02-alloydb-graph — falls back to
# the primary user if absent. In production *always* use a least-privileged role.
RO_USER = gcp.get('alloydb_ro_user') or DB_USER
RO_PASSWORD_SECRET = gcp.get('alloydb_ro_password_secret') or DB_PASSWORD_SECRET

print(f'project={PROJECT_ID} region={REGION} text2sql={TEXT2SQL_MODEL} grounding={GROUNDING_MODEL}')
print(f'alloydb_instance={INSTANCE_URI or "(not set — run 02-alloydb-graph.ipynb)"}')
print(f'read-only user={RO_USER}')

## Step 1: Open a read-only AlloyDB connection

We open a single psycopg connection through the AlloyDB Connector with the read-only role. The connection is `autocommit=True` so we never accidentally start a write transaction; we still rely on the validator to reject DDL/DML.

In [ ]:
from google.cloud.alloydb.connector import Connector
from google.cloud.secretmanager import SecretManagerServiceClient
import psycopg

assert INSTANCE_URI, 'AlloyDB instance not configured. Run 02-alloydb-graph.ipynb first.'

_sm = SecretManagerServiceClient()
_secret_path = f'projects/{PROJECT_ID}/secrets/{RO_PASSWORD_SECRET}/versions/latest'
RO_PASSWORD = _sm.access_secret_version(request={'name': _secret_path}).payload.data.decode()

_connector = Connector()

def open_ro() -> 'psycopg.Connection':
    conn = _connector.connect(
        INSTANCE_URI, 'psycopg',
        user=RO_USER, password=RO_PASSWORD, db=DB_NAME, ip_type='PUBLIC',
    )
    conn.autocommit = True
    return conn

ro_conn = open_ro()
with ro_conn.cursor() as cur:
    cur.execute('SELECT current_user, current_database();')
    print(f'✓ AlloyDB connected as {cur.fetchone()}')

## Step 2: Build a schema digest from `information_schema`

The digest is what we hand to Gemini — short, deterministic, and limited to the schemas/tables we want to expose. Excluding `pg_catalog` and `information_schema` matters: without that filter the model will sometimes invent joins to `pg_class`.

In [ ]:
ALLOWED_SCHEMAS = ('public',)
# Tables the user is allowed to query through this surface. Anything outside
# this list is rejected by the validator regardless of what the model emits.
ALLOWED_TABLES = gcp.get('text2sql_allowed_tables') or [
    'people', 'teams', 'team_members', 'skills', 'person_skills', 'projects'
]

def build_schema_digest(conn) -> str:
    sql = """
        SELECT c.table_schema, c.table_name, c.column_name, c.data_type
        FROM   information_schema.columns c
        WHERE  c.table_schema = ANY(%(schemas)s)
          AND  c.table_name = ANY(%(tables)s)
        ORDER BY c.table_schema, c.table_name, c.ordinal_position;
    """
    rows = []
    with conn.cursor() as cur:
        cur.execute(sql, {'schemas': list(ALLOWED_SCHEMAS), 'tables': list(ALLOWED_TABLES)})
        rows = cur.fetchall()
    by_table: dict[str, list[str]] = {}
    for schema, table, column, dtype in rows:
        by_table.setdefault(f'{schema}.{table}', []).append(f'{column} {dtype}')
    lines = []
    for table, cols in by_table.items():
        lines.append(f'{table}({", ".join(cols)})')
    return '\n'.join(lines)

SCHEMA_DIGEST = build_schema_digest(ro_conn)
print(f'Schema digest ({len(SCHEMA_DIGEST.splitlines())} tables, {len(SCHEMA_DIGEST)} chars):')
print(SCHEMA_DIGEST or '(empty — make sure 02-alloydb-graph.ipynb has seeded the public schema)')

## Step 3: Generate SQL with Gemini structured output

`response_mime_type='application/json'` + `response_schema` forces the model to return a parseable object. We constrain the dialect (PostgreSQL), require `SELECT/WITH` only, and ask the model to use a `:question_param`-style placeholder for any user-supplied literal — though for this demo we pass the question fully into the prompt, and rely on the validator for the safety net.

In [ ]:
from google import genai
from google.genai import types as genai_types

_client = genai.Client(vertexai=True, project=PROJECT_ID, location=REGION)

RESPONSE_SCHEMA = {
    'type': 'object',
    'properties': {
        'sql': {'type': 'string', 'description': 'A single PostgreSQL SELECT or WITH statement.'},
        'warnings': {
            'type': 'array',
            'items': {'type': 'string'},
            'description': 'Anything the user should know: column not found, ambiguous join, etc.',
        },
    },
    'required': ['sql', 'warnings'],
}

def generate_sql(question: str) -> dict:
    prompt = (
        'You translate analyst questions into PostgreSQL queries against an OrgIQ AlloyDB schema.\n'
        '\n'
        'Schema (allow-listed; nothing else exists):\n'
        f'{SCHEMA_DIGEST}\n'
        '\n'
        'Rules:\n'
        '- Output ONE statement. SELECT or WITH only. No INSERT/UPDATE/DELETE/DDL.\n'
        '- PostgreSQL 16 dialect.\n'
        '- Only reference tables and columns from the schema above.\n'
        '- Prefer explicit joins and aliases; never use SELECT *.\n'
        '- Cap result sets at 100 rows with LIMIT 100.\n'
        '- If the question is ambiguous, add a warning string; still emit your best-effort SQL.\n'
        '\n'
        f'Question: {question}\n'
    )
    resp = _client.models.generate_content(
        model=TEXT2SQL_MODEL,
        contents=prompt,
        config=genai_types.GenerateContentConfig(
            temperature=0.0,
            max_output_tokens=1024,
            response_mime_type='application/json',
            response_schema=RESPONSE_SCHEMA,
        ),
    )
    # When response_schema is set, .text is the JSON body.
    return json.loads(resp.text)

QUESTION = 'Who are the top five people by number of skills, and what is their department?'
candidate = generate_sql(QUESTION)
print('Generated candidate:')
print(json.dumps(candidate, indent=2))

## Step 4: Validate before executing

Three checks, in order of cost (same shape as the AWS notebook):

1. Single statement only — no stacked queries.
2. Starts with `SELECT` or `WITH` — and no forbidden DDL/DML keywords anywhere.
3. Every table reference is in the allow-list.

The read-only role is the second line of defence (AlloyDB will refuse `INSERT` on a `pg_read_all_data` role). The validator catches it earlier so we don't burn a round trip.

In [ ]:
import re

FORBIDDEN = re.compile(
    r'\b(INSERT|UPDATE|DELETE|DROP|ALTER|CREATE|TRUNCATE|GRANT|REVOKE|COPY|MERGE|VACUUM|CALL)\b',
    re.IGNORECASE,
)
_IDENT = re.compile(r'\b([a-z_][a-z0-9_]*)\b', re.IGNORECASE)

class SqlValidationError(Exception):
    pass

def validate_sql(sql: str, allowed_tables: list[str]) -> str:
    stripped = sql.strip().rstrip(';').strip()
    if ';' in stripped:
        raise SqlValidationError('multiple statements detected — only one query permitted')
    head = stripped.lstrip('(').split(None, 1)[0].upper()
    if head not in ('SELECT', 'WITH'):
        raise SqlValidationError(f'statement must start with SELECT or WITH, got {head!r}')
    if FORBIDDEN.search(stripped):
        raise SqlValidationError('statement contains a forbidden DDL/DML keyword')
    allowed = {t.lower() for t in allowed_tables}
    # Collect all bare identifiers that follow FROM/JOIN — best-effort, no full AST.
    refs = re.findall(r'\b(?:from|join)\s+([a-z_][a-z0-9_\.]*)', stripped, re.IGNORECASE)
    referenced = {r.split('.')[-1].lower() for r in refs}
    illegal = referenced - allowed
    if illegal:
        raise SqlValidationError(f'references disallowed tables: {sorted(illegal)}')
    if not referenced:
        raise SqlValidationError('no allow-listed tables referenced')
    return stripped

try:
    safe_sql = validate_sql(candidate['sql'], ALLOWED_TABLES)
    print('✓ SQL passed validation\n')
    print(safe_sql)
    if candidate.get('warnings'):
        print('\nModel warnings:')
        for w in candidate['warnings']:
            print(f'  - {w}')
except SqlValidationError as exc:
    print(f'✗ Validation failed: {exc}')
    raise

## Step 5: Execute against AlloyDB (read-only)

In [ ]:
def run_select(sql: str, params: dict | None = None, row_cap: int = 100) -> list[dict]:
    with ro_conn.cursor() as cur:
        cur.execute(sql, params or {})
        cols = [d.name for d in cur.description]
        rows = cur.fetchmany(row_cap)
    return [dict(zip(cols, r)) for r in rows]

rows = run_select(safe_sql)
print(f'✓ {len(rows)} row(s) returned')
print(json.dumps(rows[:10], indent=2, default=str))

## Step 6: Surface a grounded answer (cheap model)

Hand the validated rows to `gemini-2.5-flash` — the strong reasoning model already did the work in Step 3, all we need now is a one-paragraph natural-language summary. Splitting this off from the SQL step (a) is cheaper and (b) makes the grounding visible: the summary cell sees only validated rows, not the raw user question paired with a SQL prompt.

In [ ]:
def summarise(question: str, rows: list[dict]) -> str:
    prompt = (
        'Answer the analyst question using ONLY the rows below. If the rows do not\n'
        'support a clean answer, say so. Keep it under three sentences.\n'
        '\n'
        f'Question: {question}\n'
        f'Rows (JSON): {json.dumps(rows[:20], default=str)}\n'
    )
    resp = _client.models.generate_content(
        model=GROUNDING_MODEL,
        contents=prompt,
        config=genai_types.GenerateContentConfig(temperature=0.2, max_output_tokens=300),
    )
    return resp.text or ''

answer = summarise(QUESTION, rows)
print('Answer:\n')
print(answer)

## Step 7: Try a question the validator must reject

Sanity check: an injection attempt or DDL request should never reach AlloyDB.

In [ ]:
ADVERSARIAL_CASES = [
    'SELECT 1; DROP TABLE people;',                   # stacked statement
    'DELETE FROM people WHERE 1=1;',                  # DDL/DML
    'SELECT * FROM pg_catalog.pg_user;',              # not in allow-list
]
for sql in ADVERSARIAL_CASES:
    try:
        validate_sql(sql, ALLOWED_TABLES)
        print(f'✗ accepted (should have rejected): {sql}')
    except SqlValidationError as exc:
        print(f'✓ rejected: {sql!r:60s} → {exc}')

## Summary

- Schema digest built from `information_schema` against an allow-list
- Gemini 2.5 Pro generated SQL via `response_schema` (typed JSON `{sql, warnings}`)
- Validator enforces SELECT/WITH-only, no DDL/DML, allow-listed tables
- Execution rides a dedicated read-only AlloyDB role (least-privilege defence in depth)
- Grounded summary uses `gemini-2.5-flash` over validated rows
- Adversarial cases (stacked statements, DDL, catalog snooping) confirmed rejected

**Open question:** if `02-alloydb-graph.ipynb` hasn't created a dedicated `alloydb_ro_user`, this notebook falls back to the primary user. The TODO is to bake a `pg_read_all_data`-style role into the seed.

**Next:** `19-cleanup.ipynb`.